# GPT-500M Training — Kaggle 2×T4 DDP Edition

**Architecture:** Decoder-only GPT, ~505M params (28 layers × 1152 embd × 16 heads)  
**Data:** `open-web-math/open-web-math` streamed from HuggingFace  
**Target:** Kaggle 2×T4 (2 × 15.6 GB VRAM) via DistributedDataParallel  

### How DDP works here
- `torchrun` spawns 2 worker processes (one per GPU) by running `train_ddp.py`  
- Each process owns one GPU and processes its own micro-batches independently  
- Gradients are averaged across both GPUs via NCCL all-reduce before each optimizer step  
- Effective batch: 2 GPUs × micro_batch=4 × grad_accum=32 × 1024 = **262,144 tokens/step**  
- Only rank 0 saves checkpoints and prints eval results  

### Checkpoint compatibility
Checkpoints are cross-compatible with the single-GPU Colab notebook.  
The DDP `module.` prefix is stripped on save, so `.pt` files load cleanly on either platform.

### Flow
1. Install → 2. Check GPUs → 3. Seed checkpoint → 4. Write script → 5. Verify → 6. Launch  
Re-running cell 6 resumes automatically from the latest checkpoint.

---
### Parameter accounting
```
Token embedding    :  50257 × 1152  =   57.9M  (tied with lm_head)
Position embedding :   1024 × 1152  =    1.2M
Per transformer block (×28):
  CausalSelfAttn  :  4 × 1152²     =    5.31M
  FeedForward     :  8 × 1152²     =   10.62M
  LayerNorms ×2   :  4 × 1152      =    4608
28 blocks total                     =  447.0M
Final LayerNorm                     =    2304
──────────────────────────────────────────────
TOTAL                               = ~505.1M
```

## 1. Install Dependencies

In [ ]:
!pip install -q datasets transformers tokenizers tiktoken bitsandbytes accelerate

## 2. Environment Check

In [1]:
import torch

n_gpus = torch.cuda.device_count()
print(f"GPUs available : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")
print(f"PyTorch        : {torch.__version__}")

# assert n_gpus >= 2, (
#     "This notebook requires 2 GPUs. "
#     "Select 'GPU T4 x2' in Kaggle accelerator settings."
# )

GPUs available : 0
PyTorch        : 2.12.1+cpu


## 3. Configure Paths & Seed Checkpoint

In [ ]:
import os, shutil, glob

CKPT_DIR  = '/kaggle/input/phungpaul/philiane-505/Checkpoints'
CACHE_DIR = '/tmp/HF_Cache'   # ephemeral — fine for streaming, saves working space
SCRIPT    = '/kaggle/working/train_ddp.py'

os.makedirs(CKPT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Seed checkpoint from Kaggle Model registry (first session only) ───────────
existing = glob.glob(os.path.join(CKPT_DIR, 'ckpt_*.pt'))
if not existing:
    uploaded = glob.glob('/kaggle/input/**/ckpt_*.pt', recursive=True)
    if uploaded:
        src = sorted(uploaded)[-1]
        dst = os.path.join(CKPT_DIR, os.path.basename(src))
        print(f"Seeding checkpoint: {src} -> {dst}")
        shutil.copy(src, dst)
        print("Done. Training will resume from this checkpoint.")
    else:
        print("No uploaded checkpoint found — will start from scratch.")
else:
    print(f"Checkpoint already present: {sorted(existing)[-1]}")

print(f"\nCheckpoint dir : {CKPT_DIR}")
print(f"HF cache dir   : {CACHE_DIR}")

## 4. Write Training Script

The full training logic lives in `train_ddp.py`.  
`torchrun` will launch one copy per GPU — each knows its `LOCAL_RANK` via environment variables.

In [ ]:
import json as _json

_script_ = """import contextlib
import gc
import glob
import math
import os
import time
from dataclasses import dataclass
from typing import Iterator, Optional

import tiktoken
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as grad_ckpt
from datasets import load_dataset
from torch.nn.parallel import DistributedDataParallel as DDP

# ============================================================================
# DDP INITIALIZATION & SETUP
# ============================================================================

dist.init_process_group(backend="nccl")
LOCAL_RANK = int(os.environ["LOCAL_RANK"])
WORLD_SIZE = dist.get_world_size()
IS_MASTER = LOCAL_RANK == 0

torch.cuda.set_device(LOCAL_RANK)
device = f"cuda:{LOCAL_RANK}"

# Memory and performance optimizations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

if IS_MASTER:
    print(f"DDP ready | world_size={WORLD_SIZE}")
    for i in range(WORLD_SIZE):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  {mem:.1f} GB")

# Environment Paths
CKPT_DIR = "/kaggle/working/GPT500M/Checkpoints"
CACHE_DIR = "/tmp/HF_Cache"


# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class GPTConfig:
    # Architecture
    vocab_size: int = 50257
    block_size: int = 1024
    n_layer: int = 28
    n_head: int = 16
    n_embd: int = 1152
    dropout: float = 0.0

    # Training Hyperparameters
    # 2xT4 Layout: micro_batch=2 per GPU, grad_accum=64
    # Effective batch size = 2 GPUs * 2 * 64 * 1024 = 262,144 tokens/step
    num_epochs: int = 2
    micro_batch_size: int = 2
    grad_accum_steps: int = 64
    lr: float = 3e-4
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    warmup_steps: int = 1000
    train_steps: int = 112000
    eval_interval: int = 100
    eval_batches: int = 50
    tokenizer_name: str = "gpt2"
    max_checkpoints: int = 1
    device: str = "cuda"


cfg = GPTConfig()
cfg.device = device


# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

class CausalSelfAttention(nn.Module):

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout = cfg.dropout

        # Key, query, value projections combined into a single linear layer
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)

        # Local KV cache holders for inference acceleration
        self._cache_k: Optional[torch.Tensor] = None
        self._cache_v: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        def split_heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        if use_cache:
            if self._cache_k is not None:
                k = torch.cat([self._cache_k, k], dim=2)
                v = torch.cat([self._cache_v, v], dim=2)
            self._cache_k = k
            self._cache_v = v

        dropout_p = self.dropout if self.training else 0.0

        # High-performance FlashAttention/SDPA execution
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=None, dropout_p=dropout_p, is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

    def clear_cache(self):
        self._cache_k = None
        self._cache_v = None


class FeedForward(nn.Module):

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.ffn = FeedForward(cfg)

    def _block_fn(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x), use_cache=False)
        x = x + self.ffn(self.ln2(x))
        return x

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        # Save VRAM during training using activation checkpointing
        if self.training and not use_cache:
            return grad_ckpt.checkpoint(self._block_fn, x, use_reentrant=False)
        
        x = x + self.attn(self.ln1(x), use_cache=use_cache)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT500M(nn.Module):

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_final = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # Weight sharing / embedding tying
        self.head.weight = self.tok_emb.weight
        self._init_weights()

        if IS_MASTER:
            n = self.num_params()
            print(f"GPT-500M | {n:,} params ({n/1e6:.1f}M)")

    def _init_weights(self):
        for name, module in self.named_modules():
            if isinstance(module, nn.Linear):
                std = 0.02
                if name.endswith(("c_proj", "net.2")):
                    std = 0.02 / math.sqrt(2 * self.cfg.n_layer)
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def num_params(self) -> int:
        return sum(p.numel() for n, p in self.named_parameters() if n != "head.weight")

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None, use_cache: bool = False):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        positions = torch.arange(T, device=idx.device)
        
        x = self.drop(self.tok_emb(idx) + self.pos_emb(positions))
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        x = self.ln_final(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            
        return logits, loss

    def clear_kv_cache(self):
        for block in self.blocks:
            block.attn.clear_cache()


# ============================================================================
# CHECKPOINT UTILITIES
# ============================================================================

def save_checkpoint(model, optimizer, scaler, step, epoch, epoch_step, losses, cfg):
    raw_sd = model.module.state_dict()
    ckpt_path = os.path.join(CKPT_DIR, f"ckpt_{step:06d}.pt")
    
    torch.save({
        "step": step,
        "epoch": epoch,
        "epoch_step": epoch_step,
        "model": raw_sd,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "cfg": cfg,
        "losses": losses,
    }, ckpt_path)

    train_loss = losses["train"]
    val_loss = losses["val"]
    print(f"  checkpoint saved -> {ckpt_path} (epoch {epoch} | step {step} | train {train_loss:.4f} | val {val_loss:.4f})")

    # Rotate checkpoints — keep only max_checkpoints most recent
    all_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")), key=os.path.getmtime)
    for old in all_ckpts[: max(0, len(all_ckpts) - cfg.max_checkpoints)]:
        os.remove(old)
        print(f"  deleted old checkpoint: {old}")


def load_checkpoint(path, raw_model, optimizer, scaler):
    #NOTE: works for both single-GPU and DDP-wrapped models
    print(f"Loading checkpoint: {path}")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    cfg = ckpt["cfg"]
    cfg.device = device

    # Strip module. prefix safely
    sd = {k.replace("module.", ""): v for k, v in ckpt["model"].items()}
    raw_model.load_state_dict(sd)
    del sd, ckpt["model"]

    optimizer.load_state_dict(ckpt["optimizer"])
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)
    del ckpt["optimizer"]

    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    gc.collect()

    step = ckpt["step"]
    epoch = ckpt.get("epoch", 0)
    epoch_step = ckpt.get("epoch_step", 0)
    losses = ckpt["losses"]

    if IS_MASTER:
        train_loss = losses["train"]
        val_loss = losses["val"]
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"  resumed — global step {step} | epoch {epoch} | epoch_step {epoch_step}")
        print(f"  train loss {train_loss:.4f} | val loss {val_loss:.4f}")
        print(f"  VRAM after load: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")

    return cfg, step, epoch, epoch_step, losses


# ============================================================================
# DATA PIPELINE
# ============================================================================

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token


class TokenBuffer:
    def __init__(self, iterator, block_size: int):
        self._iter = iterator
        self._block_size = block_size
        self._buf: list = []

    def _generate(self) -> Iterator:
        for doc in self._iter:
            tokens = enc.encode_ordinary(doc["text"]) + [EOT]
            self._buf.extend(tokens)
            while len(self._buf) >= self._block_size + 1:
                chunk = self._buf[: self._block_size + 1]
                self._buf = self._buf[self._block_size + 1 :]
                yield chunk


def make_stream(epoch: int, skip_docs: int = 0):
    ds = load_dataset(
        "open-web-math/open-web-math",
        split="train",
        streaming=True,
        cache_dir=CACHE_DIR,
    )
    # Unique rank-based shuffling to separate processed stream chunks
    ds = ds.shuffle(seed=42 + epoch + LOCAL_RANK * 1000, buffer_size=10_000)
    if skip_docs > 0 and IS_MASTER:
        print(f"  fast-forwarding {skip_docs:,} documents in epoch {epoch} ...")
    if skip_docs > 0:
        ds = ds.skip(skip_docs)
    return iter(ds)


def make_val_tensors(n_val_docs: int = 500, block_size: int = 1024) -> torch.Tensor:
    if IS_MASTER:
        print(f"Building validation set from {n_val_docs} val docs ...")
    val_ds = load_dataset(
        "open-web-math/open-web-math",
        split="train",
        streaming=True,
        cache_dir=CACHE_DIR,
    )
    val_tokens = []
    for i, doc in enumerate(val_ds):
        if i >= n_val_docs:
            break
        val_tokens.extend(enc.encode_ordinary(doc["text"]) + [EOT])
    n = (len(val_tokens) // (block_size + 1)) * (block_size + 1)
    return torch.tensor(val_tokens[:n], dtype=torch.int64)


# ============================================================================
# TRAINING UTILITIES
# ============================================================================

def get_lr(step: int, cfg: GPTConfig) -> float:
    # Linear warmup phase
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / cfg.warmup_steps
    # Cosine annealing phase
    progress = (step - cfg.warmup_steps) / max(1, cfg.train_steps - cfg.warmup_steps)
    return cfg.lr * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))


@torch.no_grad()
def estimate_loss(model, val_data, cfg) -> dict:
    loss_tensor = torch.zeros(1, device=device)
    if IS_MASTER:
        model.eval()
        val_gpu = val_data.to(device)
        max_start = len(val_gpu) - cfg.block_size - 1
        losses = []
        with torch.amp.autocast("cuda"):
            for _ in range(cfg.eval_batches):
                ix = torch.randint(max_start, (cfg.micro_batch_size,))
                x = torch.stack([val_gpu[i : i + cfg.block_size] for i in ix])
                y = torch.stack([val_gpu[i + 1 : i + cfg.block_size + 1] for i in ix])
                _, loss = model(x, y)
                losses.append(loss.item())
        del val_gpu
        torch.cuda.empty_cache()
        loss_tensor[0] = sum(losses) / len(losses)
        model.train()
    
    dist.broadcast(loss_tensor, src=0)
    return {"val": loss_tensor.item()}


# ============================================================================
# INITIALIZE COMPONENT ARCHITECTURE & STATE
# ============================================================================

gc.collect()
torch.cuda.empty_cache()

raw_model = GPT500M(cfg)

# Weight decay segregation filter
decay_params = [p for n, p in raw_model.named_parameters() if p.dim() >= 2 and p.requires_grad]
no_decay_params = [p for n, p in raw_model.named_parameters() if p.dim() < 2 and p.requires_grad]

#NOTE: old version was torch.optim.AdamW([...], lr=cfg.lr, betas=(0.9, 0.95), fused=True). We're replacing it with bnb as a resort to GPU OOM

import bitsandbytes as bnb
optimizer = bnb.optim.AdamW([
    {"params": decay_params, "weight_decay": cfg.weight_decay},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=cfg.lr, betas=(0.9, 0.95))

scaler = torch.amp.GradScaler("cuda")

# Resume configuration tracking Checkpoint Check
checkpoints = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")), key=os.path.getmtime)
latest_ckpt = checkpoints[-1] if checkpoints else None

raw_model.to(device)
if latest_ckpt:
    cfg, start_step, start_epoch, start_epoch_step, loaded_losses = \
        load_checkpoint(latest_ckpt, raw_model, optimizer, scaler)
else:
    if IS_MASTER:
        print("No checkpoint found — starting from scratch.")
    start_step = 0
    start_epoch = 0
    start_epoch_step = 0
    loaded_losses = {"train": float("nan"), "val": float("nan")}

# Wrap structural layers into distributed framework AFTER state extraction loading
model = DDP(raw_model, device_ids=[LOCAL_RANK], find_unused_parameters=False)

# Dataset Scale Configuration Calculations
DOCS_PER_EPOCH = 6_300_000
STEPS_PER_EPOCH = DOCS_PER_EPOCH * 350 // (cfg.block_size + 1) // cfg.grad_accum_steps
cfg.train_steps = STEPS_PER_EPOCH * cfg.num_epochs

if IS_MASTER:
    tokens_per_step = WORLD_SIZE * cfg.micro_batch_size * cfg.grad_accum_steps * cfg.block_size
    print(f"\\nSteps per epoch       : {STEPS_PER_EPOCH:,}")
    print(f"Total steps           : {cfg.train_steps:,} ({cfg.num_epochs} epoch(s))")
    print(f"Start step            : {start_step}")
    print(f"Effective tokens/step : {tokens_per_step:,}")


# ============================================================================
# CONSTRUCT DATA PIPELINE STREAMS
# ============================================================================

val_data = make_val_tensors(n_val_docs=500, block_size=cfg.block_size) if IS_MASTER else None
if IS_MASTER:
    print(f"Val set ready — {len(val_data):,} tokens")

train_stream_iter = make_stream(
    epoch=start_epoch,
    skip_docs=start_epoch_step * cfg.grad_accum_steps * cfg.micro_batch_size,
)
train_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)
buf_gen = train_buf._generate()


# ============================================================================
# MAIN TRAINING CORE LOOP
# ============================================================================

model.train()
train_loss_acc = 0.0
t0 = time.time()

epoch = start_epoch
epoch_step = start_epoch_step

for step in range(start_step, cfg.train_steps):

    # Epoch Boundary Validation
    if epoch_step >= STEPS_PER_EPOCH:
        epoch += 1
        epoch_step = 0
        if epoch >= cfg.num_epochs:
            if IS_MASTER:
                print(f"\\nCompleted {cfg.num_epochs} epoch(s). Training done.")
            break
        if IS_MASTER:
            sep = "=" * 60
            print(f"\\n{sep}\\nStarting epoch {epoch}\\n{sep}")
        train_stream_iter = make_stream(epoch=epoch, skip_docs=0)
        train_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)
        buf_gen = train_buf._generate()

    # Learning Rate Update Schedule
    lr = get_lr(step, cfg)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    # Gradient Accumulation Logic Block
    with torch.amp.autocast("cuda"):
        for micro_step in range(cfg.grad_accum_steps):
            try:
                chunks = [next(buf_gen) for _ in range(cfg.micro_batch_size)]
            except StopIteration:
                epoch_step = STEPS_PER_EPOCH
                break

            data = torch.tensor(chunks, dtype=torch.int64)
            x = data[:, :-1].to(device)
            y = data[:, 1:].to(device)

            # Prevent intra-rank gradient syncing overhead except on final step
            is_last_micro = (micro_step == cfg.grad_accum_steps - 1)
            ctx = model.no_sync() if not is_last_micro else contextlib.nullcontext()
            
            with ctx:
                loss = model(x, y)
                loss = loss / cfg.grad_accum_steps
                scaler.scale(loss).backward()
            accum_loss += loss.item()

    # Optimization Step Processing Engine
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    scaler.step(optimizer)
    scaler.update()

    epoch_step += 1
    train_loss_acc += accum_loss

    # Continuous Log Telemetry 
    if IS_MASTER and step % 10 == 0:
        smooth_loss = train_loss_acc / max(1, step - start_step + 1)
        print(f"ep {epoch} | step {step:>6d}/{cfg.train_steps} | loss {accum_loss:.4f} | smooth {smooth_loss:.4f} | lr {lr:.2e}", end="\\r")

    # Periodic Evaluation Execution
    if (step % cfg.eval_interval == 0) or (step == cfg.train_steps - 1):
        val_losses = estimate_loss(model, val_data, cfg)
        smooth_train = train_loss_acc / max(1, step - start_step + 1)
        losses = {"train": smooth_train, "val": val_losses["val"]}

        if IS_MASTER:
            elapsed = time.time() - t0
            sep = "-" * 65
            print(f"\\n{sep}")
            print(f"EVAL  ep {epoch} | step {step:>6d} | train {losses['train']:.4f} | val {losses['val']:.4f} | {elapsed:.0f}s")
            save_checkpoint(model, optimizer, scaler, step, epoch, epoch_step, losses, cfg)
            t0 = time.time()

        # Collective Sync Barrier Core Lock
        dist.barrier()
        model.train()

if IS_MASTER:
    print("\\nTraining complete.")

dist.destroy_process_group()
"""

with open(SCRIPT, 'w') as _f:
    _f.write(_script_)
print(f"Training script written to {SCRIPT}")
print(f"Lines : {len(_script_.splitlines())}")

NameError: name 'IS_MASTER' is not defined

## 5. Verify Script

In [3]:
import ast
with open(SCRIPT) as f:
    src = f.read()
try:
    ast.parse(src)
    print("✓ Script syntax OK")
    print(f"  Lines : {len(src.splitlines())}")
    print(f"  Path  : {SCRIPT}")
except SyntaxError as e:
    print(f"✗ Syntax error: {e}")

NameError: name 'SCRIPT' is not defined

## 6. Launch Training

`torchrun --nproc_per_node=2` spawns 2 processes, one per GPU.  
Rank 0 handles all printing — output will not be duplicated.  

**To resume:** just re-run this cell. The script auto-detects the latest checkpoint.

In [ ]:
!torchrun --nproc_per_node=2 --master_port=29500 {SCRIPT}

## 7. Inspect Checkpoint

Loads the latest checkpoint metadata for a quick sanity check.  
Paste the model class here and uncomment the generation block to run inference.

In [ ]:
import os, glob, torch

CKPT_DIR = '/kaggle/working/GPT500M/Checkpoints'
checkpoints = sorted(glob.glob(os.path.join(CKPT_DIR, 'ckpt_*.pt')), key=os.path.getmtime)
assert checkpoints, "No checkpoints found — run the training cell first."

latest = checkpoints[-1]
print(f"Loading: {latest}")
ckpt = torch.load(latest, map_location='cpu', weights_only=False)

step       = ckpt['step']
epoch      = ckpt.get('epoch', 0)
train_loss = ckpt['losses']['train']
val_loss   = ckpt['losses']['val']
print(f"Step       : {step}")
print(f"Epoch      : {epoch}")
print(f"Train loss : {train_loss:.4f}")
print(f"Val loss   : {val_loss:.4f}")

# ── Generation (uncomment after pasting model classes above) ─────────────────
# import tiktoken, math
# from train_ddp import GPT500M   # or paste the class here
# device    = 'cuda:0'
# enc       = tiktoken.get_encoding('gpt2')
# gen_cfg   = ckpt['cfg']
# gen_model = GPT500M(gen_cfg).to(device)
# sd = {k.replace('module.', ''): v for k, v in ckpt['model'].items()}
# gen_model.load_state_dict(sd)
# gen_model.eval()
#
# PROMPT         = "The derivative of sin(x) with respect to x is"
# MAX_NEW_TOKENS = 256
# TEMPERATURE    = 0.8
# TOP_K          = 50
#
# ids = enc.encode_ordinary(PROMPT)
# x   = torch.tensor([ids], dtype=torch.int64).to(device)
# with torch.no_grad():
#     for _ in range(MAX_NEW_TOKENS):
#         logits, _ = gen_model(x[:, -gen_cfg.block_size:])
#         logits = logits[:, -1, :] / TEMPERATURE
#         top_k_vals, top_k_idx = torch.topk(logits, TOP_K)
#         probs = torch.softmax(top_k_vals, dim=-1)
#         next_id = top_k_idx[0, torch.multinomial(probs[0], 1)]
#         x = torch.cat([x, next_id.unsqueeze(0).unsqueeze(0)], dim=1)
# print(enc.decode(x[0].tolist()))